# Satellite Telecommand Daily Validator## Inference Only -- No RetrainingLoads the trained model and validates new daily telemetry against commands.**Inputs:** `best_model.pt`, `telemetry.csv`, `commands.csv`, `rules_*.yaml`**Outputs:** Verdict per command (EXPECTED/DEGRADED/WRONG/MISSING), JSON report

## 1. Setup

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
import torchcde, yaml, json, warnings
from pathlib import Path
from collections import defaultdict, Counter
from enum import Enum
from datetime import datetime
warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CFG = {
    'TELEMETRY_CSV': './live_data/telemetry.csv',
    'COMMANDS_CSV': './live_data/commands.csv',
    'MODEL_PATH': './best_model.pt',
    'RULES_DIR': './rules',
    'REPORT_DIR': './reports',
    'WINDOW_SIZE': 150,
    'CMD_AMPLITUDE': 3.0,
    'CMD_TAU': 80.0,
    'CMD_MAX_SIGNAL': 8.0,
    'CMD_EFFECT_WINDOW': 120,
}
Path(CFG['REPORT_DIR']).mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')


## 2. Load Trained Model

In [ ]:
class LowRankVectorField(nn.Module):
    def __init__(self, H, V, W, depth, R, dropout):
        super().__init__()
        self.H = H; self.V = V; self.R = R
        layers = []; d = H
        for _ in range(depth - 1):
            layers += [nn.Linear(d, W), nn.ReLU(), nn.Dropout(dropout)]
            d = W
        self.backbone = nn.Sequential(*layers)
        self.head_A = nn.Linear(d, H * R)
        self.head_B = nn.Linear(d, V * R)
    def forward(self, t, z):
        b = z.shape[0]; h = self.backbone(z)
        A = self.head_A(h).view(b, self.H, self.R)
        B = self.head_B(h).view(b, self.V, self.R)
        return torch.tanh(torch.bmm(A, B.transpose(1, 2)))

class SpacecraftCDE(nn.Module):
    def __init__(self, V, H, O, W, depth, R, dropout, solver='rk4', adjoint=False):
        super().__init__()
        self.H = H; self.solver = solver; self.adjoint = adjoint
        self.initial = nn.Sequential(nn.Linear(V, H), nn.ReLU(), nn.Dropout(dropout),
                                     nn.Linear(H, H), nn.Tanh())
        self.func = LowRankVectorField(H, V, W, depth, R, dropout)
        self.readout = nn.Sequential(nn.Linear(H, W), nn.ReLU(), nn.Dropout(dropout),
                                     nn.Linear(W, O))
    def forward(self, coeffs, eval_times):
        X = torchcde.CubicSpline(coeffs)
        z0 = self.initial(X.evaluate(X.interval[0]))
        t = eval_times.float().clamp(X.interval[0], X.interval[-1])
        z = torchcde.cdeint(X=X, func=self.func, z0=z0, t=t, adjoint=self.adjoint, method=self.solver)
        b, nt, _ = z.shape
        p = self.readout(z.reshape(-1, self.H)).reshape(b, nt, -1)
        return p - p[:, 0:1, :]

ckpt = torch.load(CFG['MODEL_PATH'], map_location=DEVICE, weights_only=False)
cfg_t = ckpt.get('config', {})
param_names = ckpt['param_names']
command_types = ckpt['command_types']
telem_mean = ckpt['telem_mean']
telem_std = ckpt['telem_std']
discrete_params = ckpt.get('discrete_params', [])
analog_params = ckpt.get('analog_params', [])
P = len(param_names); M = len(command_types); TOTAL_CH = 1 + P + M
WS = CFG['WINDOW_SIZE']

model = SpacecraftCDE(TOTAL_CH, cfg_t.get('HIDDEN_DIM',48), P, cfg_t.get('VF_WIDTH',48),
    cfg_t.get('VF_DEPTH',3), cfg_t.get('LOW_RANK',6), cfg_t.get('DROPOUT',0.15)).to(DEVICE)
model.load_state_dict(ckpt['model_state']); model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params, epoch {ckpt["epoch"]+1}, val={ckpt["val_loss"]:.4f}')


## 3. Load Rules

In [ ]:
rules = {}; RULE_EFFECTS = {}; RULE_WINDOWS = {}; GROUND_TRUTH = {}
rules_dir = Path(CFG['RULES_DIR'])
if rules_dir.exists():
    for yf in sorted(rules_dir.glob('*.yaml')) + sorted(rules_dir.glob('*.yml')):
        try:
            with open(yf) as f: rd = yaml.safe_load(f)
            sub = rd.get('subsystem', yf.stem); rules[sub] = rd
            for cn, cd in rd.get('command_effects', {}).items():
                RULE_EFFECTS.setdefault(cn, {})
                RULE_WINDOWS[cn] = cd.get('effect_window_sec', 30)
                for p, e in cd.get('must_change', {}).items():
                    RULE_EFFECTS[cn][p] = e
                    d = e.get('direction', '?'); s = +1 if d in ('increase','toggle') else -1
                    GROUND_TRUTH.setdefault(cn, {})[p] = {'value': s*(e.get('min_magnitude',0)+e.get('max_magnitude',0))/2,
                        'direction': d, 'is_toggle': d=='toggle'}
            print(f'  {sub}: {len(rd.get("command_effects",{}))} commands')
        except Exception as e: print(f'  Failed: {yf}: {e}')
print(f'Rules: {len(RULE_EFFECTS)} commands, {sum(len(v) for v in RULE_EFFECTS.values())} effects')


## 4. Load Daily Data

In [ ]:
tl = pd.read_csv(CFG['TELEMETRY_CSV'])
assert {'timestamp','parameter_name','value'}.issubset(tl.columns)
tl['timestamp'] = pd.to_datetime(tl['timestamp'])
tw = tl.pivot_table(index='timestamp', columns='parameter_name', values='value', aggfunc='mean').sort_index().ffill().bfill()
for p in param_names:
    if p not in tw.columns: tw[p] = 0.0
tw = tw[param_names]

cmds = pd.read_csv(CFG['COMMANDS_CSV'])
assert {'timestamp','command_name'}.issubset(cmds.columns)
cmds['timestamp'] = pd.to_datetime(cmds['timestamp'])
cmds = cmds.sort_values('timestamp').reset_index(drop=True)

N = len(tw); dt_sec = (tw.index[1]-tw.index[0]).total_seconds() if N > 1 else 10.0
telem_vals = tw.values.astype(np.float32)
telem_norm = (telem_vals - telem_mean) / telem_std
ti = tw.index; param_idx = {p: i for i, p in enumerate(param_names)}
disc_set = set(discrete_params)

# Command encoding
ci2 = {c: i for i, c in enumerate(command_types)}
cs = np.zeros((N, M), np.float32)
for _, r in cmds.iterrows():
    ix = min(ti.searchsorted(r['timestamp']), N-1)
    cl = ci2.get(r['command_name'])
    if cl is not None:
        for t in range(ix, min(ix+400, N)):
            cs[t, cl] += CFG['CMD_AMPLITUDE'] * np.exp(-(t-ix)/CFG['CMD_TAU'])
cs = np.clip(cs, 0, CFG['CMD_MAX_SIGNAL'])

tsec = (tw.index - tw.index[0]).total_seconds().values
tnorm = (tsec / max(tsec.max(), 1e-6)).astype(np.float32)
X = np.concatenate([tnorm.reshape(-1,1), telem_norm, cs], axis=1).astype(np.float32)
full_coeffs = torchcde.hermite_cubic_coefficients_with_backward_differences(torch.tensor(X).unsqueeze(0))
print(f'Data: {N:,} timestamps, {len(cmds)} commands, splines: {full_coeffs.shape}')


## 5. Validate Commands

In [ ]:
print('='*70)
print(f'DAILY VALIDATION: {len(cmds)} commands')
print('='*70)

results = []
for _, cmd_row in cmds.iterrows():
    cn = cmd_row['command_name']; cts = cmd_row['timestamp']
    cidx = min(ti.searchsorted(cts), N-1)
    ws = max(1, int(RULE_WINDOWS.get(cn, 30) / dt_sec))

    res = {'timestamp': cts.isoformat(), 'command': cn, 'verdicts': [], 'cde_residual': 0.0}

    # Rule evaluation
    if cn in RULE_EFFECTS:
        pre = np.mean(telem_vals[max(0,cidx-3):cidx+1], axis=0) if cidx > 0 else telem_vals[0]
        post = telem_vals[cidx:min(N, cidx+ws+1)]
        for p, spec in RULE_EFFECTS[cn].items():
            pi = param_idx.get(p)
            if pi is None: continue
            d = spec.get('direction','?'); mn = spec.get('min_magnitude',0); mx = spec.get('max_magnitude',mn)
            if len(post) > 0:
                deltas = post[:, pi] - pre[pi]
                m = float(deltas[np.argmax(np.abs(deltas))]) if d == 'toggle' else (float(np.max(deltas)) if d == 'increase' else float(np.min(deltas)))
            else: m = 0.0
            am = abs(m)
            if d == 'toggle':
                v = 'EXPECTED' if am >= mn*0.5 else ('DEGRADED' if am > 0.05 else 'MISSING')
            elif d in ('increase','decrease'):
                ok = (m > 0.05) if d == 'increase' else (m < -0.05)
                if not ok and am < 0.05: v = 'MISSING'
                elif not ok: v = 'WRONG'
                elif am >= mn * 0.5: v = 'EXPECTED'
                else: v = 'DEGRADED'
            else: v = 'EXPECTED' if am > 0.05 else 'MISSING'
            res['verdicts'].append({'param': p, 'verdict': v, 'measured': round(m,4), 'direction': d, 'discrete': p in disc_set})

    # CDE residual
    if cidx >= 20 and cidx + WS < N:
        s = max(0, cidx-20)
        wc = full_coeffs[:, s:s+WS].to(DEVICE)
        ne = min(WS-2, 120); et = torch.arange(0, ne, dtype=torch.float32).to(DEVICE)
        with torch.no_grad(): pred = model(wc, et).cpu().numpy()[0]
        actual = telem_norm[s:s+ne] - telem_norm[s]
        res['cde_residual'] = round(float(np.max(np.abs((actual-pred)*telem_std))), 4)

    vlist = [v['verdict'] for v in res['verdicts']]
    res['overall'] = 'WRONG' if 'WRONG' in vlist else ('MISSING' if 'MISSING' in vlist else ('DEGRADED' if 'DEGRADED' in vlist else ('EXPECTED' if vlist else 'NO_RULE')))

    sym = {'EXPECTED':'OK','DEGRADED':'!!','WRONG':'XX','MISSING':'??','NO_RULE':'--'}
    print(f'  [{sym.get(res["overall"],"??")}] {cts.strftime("%H:%M:%S")} {cn:35s} -> {res["overall"]:10s} (CDE: {res["cde_residual"]:.2f})')
    results.append(res)

vc = Counter(r['overall'] for r in results)
print(f'\nSummary: {dict(vc)}')


## 6. Generate Report

In [ ]:
today = datetime.utcnow().strftime('%Y%m%d')
report = {'date': today, 'model_epoch': ckpt['epoch']+1, 'n_commands': len(cmds),
    'summary': dict(vc), 'commands': results}

rpath = Path(CFG['REPORT_DIR']) / f'validation_{today}.json'
with open(rpath, 'w') as f: json.dump(report, f, indent=2, default=str)

rows = [{'timestamp':r['timestamp'],'command':r['command'],'overall':r['overall'],
    'param':v['param'],'verdict':v['verdict'],'measured':v['measured'],'cde_residual':r['cde_residual']}
    for r in results for v in r.get('verdicts',[])]
cpath = Path(CFG['REPORT_DIR']) / f'validation_{today}.csv'
pd.DataFrame(rows).to_csv(cpath, index=False)

print(f'Reports: {rpath}, {cpath}')

critical = [r for r in results if r['overall'] in ('WRONG','MISSING')]
if critical:
    print(f'\nATTENTION: {len(critical)} commands need review!')
    for r in critical:
        print(f'  [{r["overall"]}] {r["timestamp"]} {r["command"]}')
        for v in r['verdicts']:
            if v['verdict'] in ('WRONG','MISSING'):
                print(f'    -> {v["param"]}: {v["verdict"]} (measured={v["measured"]})')
else:
    print('All commands nominal.')


## 7. Quick Status

In [ ]:
n_ok=vc.get('EXPECTED',0); n_d=vc.get('DEGRADED',0); n_w=vc.get('WRONG',0); n_m=vc.get('MISSING',0)
status = 'ALL NOMINAL' if n_w==0 and n_m==0 else 'REVIEW REQUIRED'
print(f'[{today}] {status}: {n_ok}/{len(results)} OK, {n_d} degraded, {n_w} wrong, {n_m} missing')
